# 02_02 Special tokens

*📖 Book: section 2.4*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → [numbers] → model reads → model learns → model writes → MCP tool
> ```
>
> You are here: **numbers**
>
> Project files this connects to:
>
> - `src/homer_ai/tokenizer.py`
> - `src/homer_ai/train_homer.py`
>
> **This notebook:** Two gaps in the tokenizer: characters it has never seen (like `!`) were silently thrown away, and nothing marks where one book ends and the next begins.
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- Why a tokenizer that skips unknown letters is a problem
- How `<|unk|>` stands in for letters we've never seen
- Why a line break isn't enough to show where a book ends
- How `<|endoftext|>` is put between books

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **`<\|unk\|>`** | "unknown": stands in for any character the tokenizer has never seen | `!` → `<\|unk\|>` |
| **`<\|endoftext\|>`** | "one text ends here, a new one starts" | between Book 1 and Book 2 |

Words you've met before: **special token** ([02_00](../02_text_to_numbers/02_00_chapter_overview.ipynb)).

## 3. The idea

### ❓ What happens to a letter the vocabulary doesn't have?

Say the tokenizer just skips what it doesn't know. Encode `Ζεὺς!` ("Zeus!") and decode it again:

```
Ζεὺς!  →  [11, 30, 115, 43]  →  Ζεὺς
```

Five characters went in, but only four numbers came out. The `!` is gone, and there was no warning.

### ❓ Why is that so bad?

The model gets a different sentence from the one you typed, and nobody can tell. It's like a translator who skips every word they don't know and doesn't mention it.

### ❓ So what's the fix?

Add one extra token to the vocabulary: `<|unk|>`, short for **unknown**. Any letter we don't know becomes `<|unk|>` and doesn't vanish:

```
Ζεὺς!  →  [11, 30, 115, 43, 135]  →  Ζεὺς<|unk|>
```

The `!` itself is still lost, but at least you can see that something was there. In code: look each character up, and if it isn't in the vocabulary, use `<|unk|>`'s number. The `<| |>` brackets are there so it can't be mistaken for real Greek.

*Checking input is a big part of fintech work: an ID that's one letter off has to be caught, not quietly dropped. Same idea here with `<|unk|>`.*

### ❓ Why does `<|unk|>` get the number 135?

Book 1 has 135 different characters, numbered 0 to 134. We add `<|unk|>` at the end, so it gets the next free number, 135. Putting it at the end means no letter's number has to change.

### ❓ And the second problem: where does a book end?

When books are glued together, the last line of Book 1 runs straight into the first line of Book 2, and the model thinks it's all one story. A line break can't mark the join, because Book 1 alone already has 444 of them, one per verse. So we add a second special token, `<|endoftext|>`, meaning "one text ends here, a new one starts". It gets the number 136.

### ❓ Why not just type `<|endoftext|>` into the text between books?

Because our tokenizer cuts text into single characters. It would see `<`, `|`, `e`, `n`, `d`… 13 separate characters, not one signal. So we encode each book on its own and put the marker's **number** between them:

```
Book 1 → [ … ]   +   [136]   +   Book 2 → [ … ]
```

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ Which numbers do the special tokens get in a tiny vocabulary?

Here's our real `CharTokenizer`, built from just the text `"abc"`.

👉 **Predict first:** `a`, `b`, `c` take 0, 1, 2. What will `<|unk|>` get?

In [ ]:
from homer_ai.tokenizer import CharTokenizer

t = CharTokenizer("abc")
print(t.char_to_id)
print("unk_id:", t.unk_id)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`<|unk|>` comes straight after the real letters, so it gets 3. And `<|endoftext|>` gets 4.

</details>

### ❓ What happens to an unknown letter now?

👉 **Predict first:** `z` isn't in `"abc"`. What will `"abz"` encode to?

In [ ]:
ids = t.encode("abz")
print(ids)
print(t.decode(ids))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`[0, 1, 3]`, which decodes to `ab<|unk|>`. Nothing vanished.

</details>

### ❓ How many pieces would the marker break into if we typed it?

Our tokenizer cuts text into single characters, so it would see the marker as a string of characters.

👉 **Predict first:** how many?

In [ ]:
len(list("<|endoftext|>"))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

13. That's why we insert the marker's number and not its text.

</details>

### ❓ How do we put the marker between two books?

Keep one flat list of numbers. For each book except the first, add the marker first. Then add the book's numbers.

```
book 1: a b          book 2: b a
   ↓                     ↓
[ a b ]  +  [<|endoftext|>]  +  [ b a ]
```

👉 **Predict first:** with the books `"ab"` and `"ba"`, what numbers come out?

In [ ]:
books = ["ab", "ba"]
t2 = CharTokenizer("\n".join(books))

ids = []
for book in books:
    if ids:  # not the first book: mark where the previous one ended
        ids.append(t2.eot_id)
    ids.extend(t2.encode(book))

print(ids)
print(t2.decode(ids))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`[1, 2, 4, 2, 1]`: book 1, the marker (4), book 2. The marker only goes **between** books, because nothing comes after the last one.

But why is `a` number 1 and not 0? The vocabulary was built from `"\n".join(books)`, which put a line break (`\n`) between the books. The line break sorts first, so it took ID 0.

</details>

## 5. Full example

The real `load_data` (in `src/homer_ai/train_homer.py`) does all of this on the Odyssey.

👉 **Predict first:** we only have Book 1 so far. How many `<|endoftext|>` markers will the data have?

In [ ]:
from homer_ai.train_homer import BOOKS_DIR, load_data

data, tokenizer, text = load_data(BOOKS_DIR)

print("vocabulary size:", tokenizer.vocab_size)
print("<|unk|> ID:", tokenizer.unk_id)
print("<|endoftext|> ID:", tokenizer.eot_id)
print("markers in the data:", (data == tokenizer.eot_id).sum().item())

ids = tokenizer.encode("Ζεὺς!")
print(ids, "->", tokenizer.decode(ids))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

137 tokens, with `<|unk|>` = 135, `<|endoftext|>` = 136, and **0** markers (there's only one book, so nothing to separate). And `Ζεὺς!` keeps a visible `<|unk|>`.

</details>

### ❓ Your turn: how much of `ὦ Homer!` survives?

Book 1 is Greek, so the tokenizer has never seen a Latin letter. Here's a short text that mixes both. Count how many of its characters come out as `<|unk|>`.

👉 **Predict first:** the text has 8 characters. How many of them does Book 1 know?

Replace each `None`, run the cell, then run the check cell under it.

In [ ]:
mixed_ids = tokenizer.encode("ὦ Homer!")
print(mixed_ids)

# 👉 do: count how many of these IDs are the unknown token
unknown_count = None

In [ ]:
from homer_ai.lessons import check

check(
    "unknown characters in 'ὦ Homer!'",
    unknown_count,
    sum(ch not in tokenizer.char_to_id for ch in "ὦ Homer!"),
    coach="Got 8? That's every character. Count only the IDs that are equal to "
    "tokenizer.unk_id (135): a list has a count() method for that.",
)

<details><summary>💡 Hint</summary>

The unknown token's number is `tokenizer.unk_id`. A list can count how often a value is in it: `[1, 5, 5].count(5)` gives 2.

</details>

<details><summary>✅ Answer</summary>

```python
mixed_ids = tokenizer.encode("ὦ Homer!")
print(mixed_ids)

unknown_count = mixed_ids.count(tokenizer.unk_id)
```

6 of the 8: `H`, `o`, `m`, `e`, `r` and `!`. Only `ὦ` and the space are known. That Latin `o` looks just like the Greek `ο`, but it's a different character, so it's unknown too. Decode the IDs and you get `ὦ <|unk|><|unk|><|unk|><|unk|><|unk|><|unk|>`: we can see that six characters were there, but not which ones.

</details>

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ Homer never uses `<|unk|>`. So what does the model learn about it?

When reading, nothing. It never sees `<|unk|>` in training, so the numbers it has for it stay random. When guessing, it learns to **never** guess `<|unk|>`, because the right answer is always a real letter. Which is what we want.

### ❓ If I add a token to the vocabulary, can I keep my trained model?

No, because the model's size depends on the vocabulary size. Imagine a model trained on just the 135 characters, with no special tokens. Its tables have 135 rows. Add `<|unk|>` and `<|endoftext|>` and the vocabulary has 137, so the saved weights don't fit any more, and the model has to be trained again from scratch (about 8 minutes for our small one).

### ❓ Could I just add `!` to the vocabulary instead?

Only if the text contains it. The vocabulary is built from the text, and Homer has no `!`, so it never gets a number. `<|unk|>` is for exactly these cases: anything the text never showed us.

### ❓ Why put the marker between books and not after the last one?

It separates two books, so it only makes sense where a new book follows.

## 6. Recap

- A tokenizer that skips unknown characters loses them silently, like the `!` in `Ζεὺς!`.
- `<|unk|>` (135) stands in for any unknown character, so nothing vanishes.
- `<|endoftext|>` (136) goes between books as a number, because a line break can't mark a book's end.

**You can now:**

- say what happens to a character the vocabulary doesn't have
- count the `<|unk|>` tokens in an encoded text
- join several books into one list of IDs, with `<|endoftext|>` between them

**Next:** [02_03: BPE](02_03_bpe.ipynb): can tokens be bigger than one letter?